# 01. Torch Compile Modes, Backends, and Graph Breaks

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Frostig et al. (2018 *Compiling machine learning programs via high-level tracing*); Ansel et al. (2024 *PyTorch 2: Faster Machine Learning Through Dynamic Python Bytecode Transformation and Graph Compilation*).
- **Purpose:** Formulate and evaluate the PyTorch 2 compiler architecture: TorchDynamo bytecode interception, AOTAutograd, TorchInductor Triton/C++ fusion, compiler modes (`default`, `reduce-overhead`, `max-autotune`), graph break diagnosis (`torch._dynamo.explain`), and branchless conditional execution.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [10. Cryptographic Verifiable AI ZKML Freivalds and Homomorphic CKKS](../11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. Dynamic Shapes Mark Dynamic and Recompilation Guards](02_dynamic_shapes_mark_dynamic_and_recompilation_guards.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Avoid `.item()`, `print(tensor)`, or data-dependent Python `if` inside compiled modules (triggers catastrophic graph breaks and dispatch stalls); do not use `mode="reduce-overhead"` if tensor memory addresses change between iterations.

## Mathematical Foundations & Governing Equations

1. **Fused SwiGLU Activation**: $y = \text{SwiGLU}(x, g) = (x \odot \sigma(x)) \odot g$
2. **Fused Add + LayerNorm**: $h = x + x_{\text{res}}, \quad y = \frac{h - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta$
3. **Branchless Conditional Selection**: $y = \text{where}(x > \tau, x \cdot s_{\text{high}}, x \cdot s_{\text{low}})$
4. **Arithmetic Intensity & Memory Traffic Reduction**: $\Delta M_{\text{DRAM}} = (3 - 1) \cdot \text{Bytes}(x)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def fused_swiglu(x, gate):
    return (x * torch.sigmoid(x)) * gate

def fused_add_layernorm(x, residual, gamma, beta, eps=1e-5):
    h = x + residual
    mean = torch.mean(h, dim=-1, keepdim=True)
    var = torch.mean((h - mean) ** 2, dim=-1, keepdim=True)
    norm = (h - mean) / torch.sqrt(var + float(eps))
    return norm * gamma + beta

def tensor_cond_select(x, threshold, scale_high, scale_low):
    return torch.where(x > float(threshold), x * float(scale_high), x * float(scale_low))

In [ ]:
# Parity verification against golden reference tensors
swiglu = fused_swiglu(weights["swiglu_x"], weights["swiglu_gate"])
assert torch.max(torch.abs(swiglu - weights["swiglu_ref"])) < 1e-5

add_ln = fused_add_layernorm(
    weights["ln_x"], weights["ln_residual"], weights["ln_gamma"], weights["ln_beta"], eps=1e-5
)
assert torch.max(torch.abs(add_ln - weights["add_ln_ref"])) < 1e-5

cond = tensor_cond_select(weights["cond_x"], threshold=0.0, scale_high=2.0, scale_low=0.5)
assert torch.max(torch.abs(cond - weights["cond_ref"])) < 1e-5
print("All SwiGLU, Add-LayerNorm, and Conditional Select assertions passed!")

In [ ]:
# Benchmarking fusion candidates vs eager execution
t0 = time.perf_counter()
for _ in range(500):
    _ = fused_swiglu(weights["swiglu_x"], weights["swiglu_gate"])
    _ = fused_add_layernorm(weights["ln_x"], weights["ln_residual"], weights["ln_gamma"], weights["ln_beta"], 1e-5)
    _ = tensor_cond_select(weights["cond_x"], 0.0, 2.0, 0.5)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined fusion & branchless select latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Graph Break vs Branchless Execution
# 1. Demonstrate branchless selection avoiding data-dependent Python if
x_test = weights["cond_x"]
out_branchless = tensor_cond_select(x_test, threshold=0.0, scale_high=2.0, scale_low=0.5)
print(f"Branchless output shape: {out_branchless.shape}")
assert out_branchless.shape == x_test.shape, "Branchless condition must preserve shape!"

# 2. Fused SwiGLU: verify non-linearity and gating activation
sig_vals = torch.sigmoid(weights["swiglu_x"])
assert torch.all(sig_vals >= 0.0) and torch.all(sig_vals <= 1.0), "Sigmoid gate must be in [0, 1]!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **TorchDynamo** intercepts bytecode at runtime, capturing computational subgraphs into FX IR without whole-program static constraints.
2. **TorchInductor** generates fused Triton and C++ kernels, collapsing multiple point-wise memory passes into a single register pass.
3. **Graph Breaks** degrade performance by bouncing back to CPython; avoid `.item()` and replace Python branches with `torch.where` or `torch.cond`.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_torch_compile_modes_backends_and_graph_breaks.md](../../markdown/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.md)
- **PyTorch 2.x Module:** [01_torch_compile_modes_backends_and_graph_breaks.py](../../pytorch2/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.py)
- **Functional JAX Module:** [01_torch_compile_modes_backends_and_graph_breaks.py](../../jax/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.py)
- **Native Rust Candle Track:** [01_torch_compile_modes_backends_and_graph_breaks.rs](../../candle/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.rs)
- **Hardware Kernel (CPU / WGSL):** [01_torch_compile_modes_backends_and_graph_breaks.rs](../../kernels/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.rs)
- **Golden SafeTensors:** [01_torch_compile_modes_backends_and_graph_breaks.safetensors](../../weights/12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 10. Cryptographic Verifiable AI ZKML Freivalds and Homomorphic CKKS](../11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 02. Dynamic Shapes Mark Dynamic and Recompilation Guards](02_dynamic_shapes_mark_dynamic_and_recompilation_guards.ipynb) &rarr;